# IBM Bob Internship Project: Employee Salary Prediction using Machine Learning

**Organization:** Bharat Cares & IBM SkillsBuild  
**Project Track:** AI & Machine Learning Compensation Forecasting  
**Author / Intern:** Pair Programming Submission  

---

## 1. Project Overview & Business Objective
Compensation determination is a core strategic priority in human resource management. Traditional salary structures often suffer from manual inconsistency, wage compression, or market misalignment.

This machine learning project builds an end-to-end predictive pipeline to estimate employee salaries accurately based on:
1. **Age** (demographic factor)
2. **Years of Professional Experience** (career tenure)
3. **Education Level** (Bachelor's, Master's, PhD)
4. **Gender** (equity & feature analysis)
5. **Job Title & Hierarchy** (Domain classification and seniority ranking)


In [ ]:
# Import fundamental data science and ML libraries
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Algorithms
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
import joblib

from model_utils import engineer_features, extract_department, extract_seniority
from predict import predict_single

# Set aesthetic styling
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 6)


## 2. Dataset Loading & Exploratory Data Analysis (EDA)

In [ ]:
# Load raw CSV file
df_raw = pd.read_csv('Salary Data.csv')
print("Raw dataset shape:", df_raw.shape)
df_raw.head()


In [ ]:
# Check for missing values and summary stats
print("Missing values summary:")
print(df_raw.isnull().sum())
print("
Numerical stats:")
df_raw.describe()


## 3. Data Cleaning & Anomaly Correction
- **Missing Rows:** We observed 2 rows with all NaN attributes (dropped).
- **Outlier Entry:** Line 261 contains Salary = 350 for 1.5 yrs experience, while corresponding record has 35,000 (scaled x100).
- **Categorical Standardization:** Standardized education level values (`Bachelor` -> `Bachelor's`).


In [ ]:
# Clean dataset
df_clean = df_raw.dropna(subset=["Age", "Salary"]).copy()
df_clean.loc[df_clean["Salary"] < 1000, "Salary"] *= 100
df_clean["Education Level"] = df_clean["Education Level"].replace({
    "Bachelor": "Bachelor's",
    "Master": "Master's",
    "phd": "PhD",
    "PHD": "PhD"
})

print(f"Cleaned dataset records: {len(df_clean)}")
print(f"Salary Range: ${df_clean['Salary'].min():,.0f} to ${df_clean['Salary'].max():,.0f}")


In [ ]:
# Exploratory Visualizations
fig, ax = plt.subplots(1, 2, figsize=(16, 6))

sns.scatterplot(
    data=df_clean,
    x="Years of Experience",
    y="Salary",
    hue="Education Level",
    style="Gender",
    s=80,
    ax=ax[0]
)
ax[0].set_title("Salary vs Experience by Education Level", fontsize=13, fontweight='bold')
ax[0].yaxis.set_major_formatter("${x:,.0f}")

sns.histplot(df_clean["Salary"], kde=True, color="#1f77b4", bins=20, ax=ax[1])
ax[1].set_title("Distribution of Salaries", fontsize=13, fontweight='bold')
ax[1].xaxis.set_major_formatter("${x:,.0f}")

plt.tight_layout()
plt.show()


## 4. Feature Engineering
We extract semantic seniority tiers and industry departments from the high-cardinality `Job Title` column, along with ratio interactions:
1. `Seniority`: Executive, Senior, Mid, Junior
2. `Department`: Engineering & IT, Data & Analytics, Sales & Marketing, Management & Operations, HR, Finance, etc.
3. `Experience_to_Age_Ratio`: Ratio of work experience to age.


In [ ]:
df_features = engineer_features(df_clean)
features = ["Age", "Gender", "Education Level", "Years of Experience", "Experience_to_Age_Ratio", "Seniority", "Department", "Job Title"]
X = df_features[features]
y = df_features["Salary"]

print("Engineered features sample:")
df_features[["Job Title", "Seniority", "Department", "Experience_to_Age_Ratio"]].head()


## 5. Model Architecture & Pipeline Setup

In [ ]:
# Split into Train and Test sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

# Build unified ColumnTransformer Preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ]),
            ["Age", "Years of Experience", "Experience_to_Age_Ratio"]
        ),
        (
            "ord_edu",
            OrdinalEncoder(
                categories=[["Bachelor's", "Master's", "PhD"]],
                handle_unknown="use_encoded_value",
                unknown_value=-1
            ),
            ["Education Level"]
        ),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore", min_frequency=2),
            ["Gender", "Seniority", "Department", "Job Title"]
        )
    ]
)

# Benchmark models dictionary
models = {
    "Random Forest": RandomForestRegressor(n_estimators=150, max_depth=12, random_state=42),
    "Extra Trees": ExtraTreesRegressor(n_estimators=120, max_depth=12, random_state=42),
    "Ridge Regression": Ridge(alpha=1.0),
    "Linear Regression": LinearRegression(),
    "Lasso Regression": Lasso(alpha=50.0),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=100, learning_rate=0.08, random_state=42),
    "K-Nearest Neighbors": KNeighborsRegressor(n_neighbors=5, weights="distance"),
    "Support Vector Regressor": SVR(C=5000, kernel="rbf"),
    "Decision Tree": DecisionTreeRegressor(max_depth=6, random_state=42)
}


## 6. Training, Cross-Validation & Metric Evaluation

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
results = []
fitted_models = {}

for name, model in models.items():
    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])
    
    cv_scores = cross_val_score(pipe, X_train, y_train, cv=kf, scoring="r2")
    pipe.fit(X_train, y_train)
    fitted_models[name] = pipe
    
    y_pred = pipe.predict(X_test)
    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100
    
    results.append({
        "Model": name,
        "CV_R2": cv_scores.mean(),
        "Test_R2": r2,
        "MAE": mae,
        "RMSE": rmse,
        "MAPE (%)": mape
    })

results_df = pd.DataFrame(results).sort_values(by="Test_R2", ascending=False).reset_index(drop=True)
results_df


## 7. Model Evaluation Visualizations & Diagnostics

In [ ]:
best_model_name = results_df.iloc[0]["Model"]
best_pipe = fitted_models[best_model_name]
y_pred_best = best_pipe.predict(X_test)
residuals = y_test - y_pred_best

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Actual vs Predicted
axes[0].scatter(y_test, y_pred_best, color="#1f77b4", alpha=0.75, edgecolors="k", s=70)
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "r--", lw=2, label="Perfect Fit (y = x)")
axes[0].set_title(f"Actual vs Predicted Salary ({best_model_name})", fontsize=13, fontweight='bold')
axes[0].set_xlabel("Actual Salary (USD)")
axes[0].set_ylabel("Predicted Salary (USD)")
axes[0].yaxis.set_major_formatter("${x:,.0f}")
axes[0].xaxis.set_major_formatter("${x:,.0f}")
axes[0].legend()

# Residuals distribution
sns.histplot(residuals, kde=True, color="#2ca02c", bins=18, ax=axes[1])
axes[1].axvline(0, color="red", linestyle="--", lw=1.5, label="Zero Error Line")
axes[1].set_title(f"Residual Error Distribution - {best_model_name}", fontsize=13, fontweight='bold')
axes[1].set_xlabel("Residual (Actual - Predicted) USD")
axes[1].xaxis.set_major_formatter("${x:,.0f}")
axes[1].legend()

plt.tight_layout()
plt.show()


## 8. Inference Demonstration on Custom Profiles

In [ ]:
sample_profiles = [
    {"Age": 26, "Gender": "Female", "Education Level": "Bachelor's", "Job Title": "Junior Data Analyst", "Years of Experience": 2.0},
    {"Age": 32, "Gender": "Male", "Education Level": "Master's", "Job Title": "Senior Software Engineer", "Years of Experience": 7.0},
    {"Age": 45, "Gender": "Male", "Education Level": "PhD", "Job Title": "Director of Operations", "Years of Experience": 18.0}
]

for profile in sample_profiles:
    res = predict_single(
        age=profile["Age"],
        gender=profile["Gender"],
        education_level=profile["Education Level"],
        job_title=profile["Job Title"],
        years_of_experience=profile["Years of Experience"],
        model=best_pipe
    )
    print(f"Role: {res['Job Title']} | Exp: {res['Years of Experience']} yrs | Edu: {res['Education Level']}")
    print(f"  -> Predicted Salary: ${res['Predicted Salary']:,.2f}")
    print(f"  -> Estimated Range:  ${res['Salary Lower Bound (95% approx)']:,.2f} - ${res['Salary Upper Bound (95% approx)']:,.2f}
")
